In [ ]:
# ==============================================================================
# Cell 1: Setup - Download GSC V2 + Clone Repo
# ==============================================================================
import os, subprocess, sys, importlib.util

# Clone / update repo
REPO = '/content/NC-KWS-FineTuning'
if os.path.exists(REPO) and os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    if os.path.exists(REPO):
        import shutil; shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', 'https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git', REPO], check=True)

os.chdir(REPO)

# Direct file import (bypass sys.path issues)
_nm_path = os.path.join(REPO, 'nanomamba.py')
assert os.path.isfile(_nm_path), f'nanomamba.py not found! Files: {os.listdir(REPO)}'
_spec = importlib.util.spec_from_file_location('nanomamba', _nm_path)
_nanomamba = importlib.util.module_from_spec(_spec)
sys.modules['nanomamba'] = _nanomamba
_spec.loader.exec_module(_nanomamba)

from nanomamba import create_nc_tcn_20k

# Download GSC V2
GSC_DIR = '/content/speech_commands_v0.02'
if not os.path.exists(GSC_DIR):
    print('Downloading Google Speech Commands V2 (~2.3GB)...')
    os.makedirs(GSC_DIR, exist_ok=True)
    subprocess.run(['wget', '-q', '--show-progress',
                    'http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz',
                    '-O', '/content/gsc_v2.tar.gz'], check=True)
    subprocess.run(['tar', 'xzf', '/content/gsc_v2.tar.gz', '-C', GSC_DIR], check=True)
    os.remove('/content/gsc_v2.tar.gz')
    print('Done!')
else:
    print(f'GSC V2 already at {GSC_DIR}')

print(f'NC-TCN-20K: {sum(p.numel() for p in create_nc_tcn_20k().parameters()):,} params')
print('Setup complete!')

In [ ]:
# ==============================================================================
# Cell 2: Standard 6-Task Incremental KWS Benchmark
# ==============================================================================
# Protocol: Following DE-KWS (ICASSP 2025) and AnalyticKWS (ACL 2025)
# - GSC V2, 35 keywords, standard train/val/test splits
# - 15 base classes (Task 0) + 5 incremental tasks x 4 keywords
# - Metrics: ACC (avg accuracy), BWT (backward transfer)
# - 3 random seeds for task assignment
# ==============================================================================

import os, sys, time, random, copy, subprocess, importlib.util
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

# ---- Auto-setup (in case Cell 1 was skipped) ----
REPO = '/content/NC-KWS-FineTuning'
if not os.path.exists(REPO) or not os.path.isdir(os.path.join(REPO, '.git')):
    if os.path.exists(REPO):
        import shutil; shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', 'https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git', REPO], check=True)
os.chdir(REPO)

# Direct file import (bypass sys.path issues in Colab)
if 'nanomamba' not in sys.modules:
    _nm_path = os.path.join(REPO, 'nanomamba.py')
    assert os.path.isfile(_nm_path), f'nanomamba.py not found! Files: {os.listdir(REPO)}'
    _spec = importlib.util.spec_from_file_location('nanomamba', _nm_path)
    _nanomamba = importlib.util.module_from_spec(_spec)
    sys.modules['nanomamba'] = _nanomamba
    _spec.loader.exec_module(_nanomamba)

from nanomamba import create_nc_tcn_20k
print(f'Repo: {REPO}, nanomamba loaded OK')

SR = 16000
GSC_DIR = '/content/speech_commands_v0.02'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# All 35 GSC V2 keywords
ALL_KEYWORDS = sorted([
    'backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five',
    'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left',
    'marvin', 'nine', 'no', 'off', 'on', 'one', 'right', 'seven',
    'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up', 'visual',
    'wow', 'yes', 'zero'
])
assert len(ALL_KEYWORDS) == 35

N_SEEDS = 3
BASE_EPOCHS = 30
INC_EPOCHS = 20

# ---- Parse standard splits ----
def parse_splits():
    val_files, test_files = set(), set()
    with open(os.path.join(GSC_DIR, 'validation_list.txt')) as f:
        for line in f:
            val_files.add(line.strip())
    with open(os.path.join(GSC_DIR, 'testing_list.txt')) as f:
        for line in f:
            test_files.add(line.strip())
    return val_files, test_files

val_files, test_files = parse_splits()
print(f'Standard splits: {len(val_files)} val, {len(test_files)} test')

# ---- Load audio helper ----
def load_wav_file(filepath):
    w, sr = torchaudio.load(filepath)
    a = w[0].numpy()
    if len(a) > SR: a = a[:SR]
    elif len(a) < SR: a = np.pad(a, (0, SR - len(a)))
    return a.astype(np.float32)

def load_keyword_data(keywords):
    train, val, test = {}, {}, {}
    for kw in keywords:
        kw_dir = os.path.join(GSC_DIR, kw)
        if not os.path.isdir(kw_dir): continue
        train[kw], val[kw], test[kw] = [], [], []
        for fname in sorted(os.listdir(kw_dir)):
            if not fname.endswith('.wav'): continue
            rel = f'{kw}/{fname}'
            fpath = os.path.join(kw_dir, fname)
            try:
                a = load_wav_file(fpath)
                if rel in test_files:
                    test[kw].append(a)
                elif rel in val_files:
                    val[kw].append(a)
                else:
                    train[kw].append(a)
            except: continue
    return train, val, test

print('Loading all 35 keywords...')
t0 = time.time()
all_train, all_val, all_test = load_keyword_data(ALL_KEYWORDS)
total = sum(len(v) for v in all_train.values()) + sum(len(v) for v in all_val.values()) + sum(len(v) for v in all_test.values())
print(f'Loaded {total} utterances in {time.time()-t0:.1f}s')
for kw in ALL_KEYWORDS[:5]:
    print(f'  {kw}: train={len(all_train.get(kw,[]))}, val={len(all_val.get(kw,[]))}, test={len(all_test.get(kw,[]))}')
print('  ...')

# ---- Audio Augmentation Suite ----
def aug_time_stretch(a, rate):
    n = int(len(a) / rate)
    indices = np.linspace(0, len(a)-1, n).astype(np.float32)
    idx_floor = np.floor(indices).astype(int)
    idx_ceil = np.minimum(idx_floor + 1, len(a)-1)
    frac = indices - idx_floor
    stretched = a[idx_floor] * (1 - frac) + a[idx_ceil] * frac
    if len(stretched) > SR: stretched = stretched[:SR]
    elif len(stretched) < SR: stretched = np.pad(stretched, (0, SR - len(stretched)))
    return stretched.astype(np.float32)

def aug_time_mask(a, max_frac=0.15):
    out = a.copy()
    ml = int(len(a) * np.random.uniform(0.05, max_frac))
    s = np.random.randint(0, len(a) - ml)
    out[s:s+ml] = 0.0
    return out

def augment_sample(a, bg_pool=None):
    out = a.copy()
    if np.random.random() < 0.5:
        out = aug_time_stretch(out, np.random.uniform(0.85, 1.15))
    if np.random.random() < 0.5:
        out = np.roll(out, np.random.randint(-2400, 2400))
    if np.random.random() < 0.5:
        db = np.random.uniform(-6, 6)
        out = out * (10 ** (db / 20))
    if np.random.random() < 0.4:
        out = aug_time_mask(out)
    if np.random.random() < 0.3 and bg_pool:
        bg = bg_pool[np.random.randint(len(bg_pool))]
        snr = np.random.uniform(5, 20)
        sp = max(np.mean(out**2), 1e-10)
        bp = max(np.mean(bg**2), 1e-10)
        out = out + bg * np.sqrt(sp/bp) * 10**(-snr/20)
    out = out + np.random.randn(len(out)).astype(np.float32) * np.random.uniform(0.001, 0.005)
    return out.astype(np.float32)

# ---- LoRA ----
class LoRALinear(nn.Module):
    def __init__(self, original, rank=4, alpha=8):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling
    def merge(self):
        with torch.no_grad():
            self.original.weight.data += (self.lora_B.t() @ self.lora_A.t()) * self.scaling
        return self.original

def label_smooth_ce(logits, targets, smoothing=0.1):
    n_cls = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)
    nll = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
    smooth = -log_probs.mean(dim=-1)
    return ((1 - smoothing) * nll + smoothing * smooth).mean()

# ---- Train base model from scratch (Task 0) ----
def train_base(keywords, train_data, val_data, epochs=30):
    n_cls = len(keywords)
    model = create_nc_tcn_20k(n_classes=n_cls).to(DEVICE)
    print(f'  Training base model: {n_cls} classes, {sum(p.numel() for p in model.parameters()):,} params')

    bg_pool = []
    X, Y = [], []
    for i, kw in enumerate(keywords):
        samples = train_data.get(kw, [])
        bg_pool.extend(samples[:20])
        for a in samples:
            X.append(a); Y.append(i)

    print(f'  Training samples: {len(X)}')
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(X))
        eloss, nb = 0, 0
        for i in range(0, len(X), 64):
            bi = perm[i:i+64]
            batch_x = []
            for j in bi:
                if np.random.random() < 0.5:
                    batch_x.append(torch.from_numpy(augment_sample(X[j], bg_pool)).float())
                else:
                    batch_x.append(torch.from_numpy(X[j]).float())
            bx = torch.stack(batch_x).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            logits = model(bx)
            loss = F.cross_entropy(logits, by)
            opt.zero_grad(); loss.backward(); opt.step()
            eloss += loss.item(); nb += 1
        sched.step()

        if (ep+1) % 10 == 0 or ep == 0:
            model.eval()
            vc, vt = 0, 0
            with torch.no_grad():
                for i, kw in enumerate(keywords):
                    for a in val_data.get(kw, []):
                        x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                        pred = model(x).argmax(-1).item()
                        vt += 1; vc += (pred == i)
            vacc = vc/max(vt,1)*100
            print(f'    Epoch {ep+1}/{epochs} loss={eloss/nb:.4f} val_acc={vacc:.1f}%')

    return model

# ---- Evaluate on all seen classes ----
def evaluate(model, keywords, test_data):
    model.eval()
    correct, total = 0, 0
    per_class = {}
    with torch.no_grad():
        for i, kw in enumerate(keywords):
            kc, kt = 0, 0
            for a in test_data.get(kw, []):
                x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                pred = model(x).argmax(-1).item()
                kt += 1; kc += (pred == i)
            per_class[kw] = kc / max(kt, 1)
            correct += kc; total += kt
    overall = correct / max(total, 1)
    return overall, per_class

# ---- NC-OPAL Incremental ----
def ncopal_increment(model, seen_kws, new_kws, train_data, epochs=25, lr=1.5e-3, lambda_kd=5.0, kd_temp=3.0):
    n_old = len(seen_kws)
    n_new = len(new_kws)
    n_total = n_old + n_new
    all_kws = seen_kws + new_kws
    d = model.classifier.in_features

    emb_hook = {}
    def hook_fn(m, inp, out): emb_hook['e'] = inp[0].detach()
    def get_emb(audio):
        h = model.classifier.register_forward_hook(hook_fn)
        with torch.no_grad(): model(torch.from_numpy(audio).float().unsqueeze(0).to(DEVICE))
        h.remove()
        return emb_hook['e'].squeeze(0)

    old_head = model.classifier
    new_head = nn.Linear(d, n_total).to(DEVICE)
    with torch.no_grad():
        new_head.weight[:n_old] = old_head.weight
        new_head.bias[:n_old] = old_head.bias
        scale = old_head.weight.norm(dim=1).mean().item()
        for i, kw in enumerate(new_kws):
            embs = [get_emb(a) for a in train_data.get(kw, [])[:30]]
            if embs:
                proto = F.normalize(torch.stack(embs).mean(0), dim=0)
                new_head.weight[n_old+i] = proto * scale
                new_head.bias[n_old+i] = old_head.bias.mean().item()
    model.classifier = new_head

    loras = []
    for block in model.blocks:
        for attr in ['in_proj', 'out_proj']:
            if hasattr(block, attr):
                lo = LoRALinear(getattr(block, attr), rank=2, alpha=4).to(DEVICE)
                setattr(block, attr, lo); loras.append(lo)

    teacher = copy.deepcopy(model)
    for block in teacher.blocks:
        for attr in ['in_proj', 'out_proj']:
            m = getattr(block, attr, None)
            if isinstance(m, LoRALinear):
                setattr(block, attr, m.original)
    teacher.classifier = old_head
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False

    bg_pool = []
    for kw in all_kws:
        bg_pool.extend(train_data.get(kw, [])[:10])
    X, Y = [], []
    for i, kw in enumerate(new_kws):
        li = n_old + i
        samples = train_data.get(kw, [])[:100]
        for a in samples:
            X.append(a); Y.append(li)
            for _ in range(8):
                X.append(augment_sample(a, bg_pool)); Y.append(li)
    for i, kw in enumerate(seen_kws):
        samples = train_data.get(kw, [])[:30]
        for a in samples:
            X.append(a); Y.append(i)
            for _ in range(4):
                X.append(augment_sample(a, bg_pool)); Y.append(i)

    n_new_samples = sum(1 for y in Y if y >= n_old)
    n_old_samples = sum(1 for y in Y if y < n_old)
    print(f'    Train: {len(X)} samples (new={n_new_samples}, old={n_old_samples})')

    for p in model.parameters(): p.requires_grad = False
    for p in new_head.parameters(): p.requires_grad = True

    opt1 = torch.optim.AdamW(new_head.parameters(), lr=lr*3, weight_decay=0.01)
    model.train()
    for ep in range(5):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), 32):
            bi = perm[i:i+32]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = label_smooth_ce(model(bx), by, 0.1)
            opt1.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(new_head.parameters(), 1.0)
            opt1.step()
    print(f'    Stage 1 (head-only) done')

    for m in loras:
        for p in m.parameters(): p.requires_grad = True

    opt2 = torch.optim.AdamW([
        {'params': [p for m in loras for p in m.parameters()], 'lr': lr},
        {'params': new_head.parameters(), 'lr': lr * 0.5},
    ], weight_decay=0.01)
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs-5)

    for ep in range(epochs - 5):
        perm = np.random.permutation(len(X))
        ecl, ekd, nbt = 0, 0, 0
        for i in range(0, len(X), 32):
            bi = perm[i:i+32]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            logits = model(bx)
            cls_loss = label_smooth_ce(logits, by, 0.05)

            kd_loss = torch.tensor(0.0).to(DEVICE)
            old_mask = by < n_old
            if old_mask.any():
                with torch.no_grad(): t_logits = teacher(bx[old_mask])
                s_base = logits[old_mask][:, :n_old] / kd_temp
                t_base = t_logits / kd_temp
                kd_loss = F.kl_div(F.log_softmax(s_base, -1),
                                   F.softmax(t_base, -1),
                                   reduction='batchmean') * (kd_temp**2)

            kd_w = lambda_kd * min(1.0, ep / 5.0)
            total = cls_loss + kd_w * kd_loss
            opt2.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt2.step()
            ecl += cls_loss.item(); ekd += kd_loss.item(); nbt += 1
        sched2.step()
        if (ep+1) % 5 == 0:
            print(f'      Stage 2 ep {ep+1}/{epochs-5} cls={ecl/nbt:.3f} kd={ekd/nbt:.3f}')

    for block in model.blocks:
        for attr in ['in_proj', 'out_proj']:
            m = getattr(block, attr, None)
            if isinstance(m, LoRALinear):
                setattr(block, attr, m.merge())

    return model

# ---- Finetune baseline ----
def finetune_increment(model, seen_kws, new_kws, train_data, epochs=20, lr=3e-3):
    n_old = len(seen_kws)
    n_new = len(new_kws)
    n_total = n_old + n_new
    all_kws = seen_kws + new_kws
    d = model.classifier.in_features

    old_head = model.classifier
    new_head = nn.Linear(d, n_total).to(DEVICE)
    with torch.no_grad():
        new_head.weight[:n_old] = old_head.weight
        new_head.bias[:n_old] = old_head.bias
        nn.init.xavier_uniform_(new_head.weight[n_old:])
        new_head.bias[n_old:].zero_()
    model.classifier = new_head

    bg_pool = []
    for kw in all_kws:
        bg_pool.extend(train_data.get(kw, [])[:10])
    X, Y = [], []
    for i, kw in enumerate(new_kws):
        li = n_old + i
        samples = train_data.get(kw, [])[:100]
        for a in samples:
            X.append(a); Y.append(li)
            for _ in range(8):
                X.append(augment_sample(a, bg_pool)); Y.append(li)
    for i, kw in enumerate(seen_kws):
        for a in train_data.get(kw, [])[:15]:
            X.append(a); Y.append(i)
            for _ in range(4):
                X.append(augment_sample(a, bg_pool)); Y.append(i)

    print(f'    Train: {len(X)} samples')

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    model.train()
    for ep in range(epochs):
        perm = np.random.permutation(len(X))
        eloss, nb = 0, 0
        for i in range(0, len(X), 32):
            bi = perm[i:i+32]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = F.cross_entropy(model(bx), by)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            eloss += loss.item(); nb += 1
        sched.step()
        if (ep+1) % 5 == 0:
            print(f'      Epoch {ep+1}/{epochs} loss={eloss/nb:.4f}')
    return model

# ==========================================
# MAIN BENCHMARK LOOP
# ==========================================
print('\n' + '='*70)
print('  STANDARD 6-TASK INCREMENTAL KWS BENCHMARK')
print('  Protocol: DE-KWS (ICASSP 2025) / AnalyticKWS (ACL 2025)')
print('  Model: NC-TCN-20K (21,689 params)')
print(f'  Device: {DEVICE}')
print('='*70)

all_results = {}

for method_name in ['Finetune', 'NC-OPAL']:
    print(f'\n{"="*60}')
    print(f'  METHOD: {method_name}')
    print(f'{"="*60}')

    seed_accs = []
    seed_bwts = []

    for seed in range(N_SEEDS):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed(seed)

        kws = ALL_KEYWORDS.copy()
        random.shuffle(kws)
        task_0_kws = kws[:15]
        inc_tasks = [kws[15+i*4:15+(i+1)*4] for i in range(5)]

        print(f'\n  --- Seed {seed} ---')
        print(f'  Task 0 ({len(task_0_kws)}): {task_0_kws[:5]}...')
        for t, tk in enumerate(inc_tasks):
            print(f'  Task {t+1} ({len(tk)}): {tk}')

        print(f'\n  Training base model (Task 0)...')
        base_model = train_base(task_0_kws, all_train, all_val, epochs=BASE_EPOCHS)

        acc_0, _ = evaluate(base_model, task_0_kws, all_test)
        print(f'  Task 0 accuracy: {acc_0*100:.1f}%')

        acc_after_task = [acc_0]

        model = copy.deepcopy(base_model)
        seen = task_0_kws.copy()

        for t, new_kws in enumerate(inc_tasks):
            t_start = time.time()
            print(f'\n  Task {t+1}: Adding {new_kws}')

            if method_name == 'NC-OPAL':
                model = ncopal_increment(model, seen, new_kws, all_train, epochs=INC_EPOCHS)
            else:
                model = finetune_increment(model, seen, new_kws, all_train, epochs=INC_EPOCHS)

            seen = seen + new_kws
            acc_t, per_cls = evaluate(model, seen, all_test)
            acc_after_task.append(acc_t)
            elapsed = time.time() - t_start
            print(f'    Accuracy ({len(seen)} classes): {acc_t*100:.1f}% [{elapsed:.0f}s]')

        ACC = np.mean(acc_after_task) * 100
        final_acc = acc_after_task[-1]
        BWT_vals = []
        for t in range(len(acc_after_task)-1):
            BWT_vals.append(final_acc - acc_after_task[t])
        BWT = np.mean(BWT_vals) * 100 if BWT_vals else 0

        print(f'\n  Seed {seed} Results:')
        print(f'    Acc per task: {[f"{a*100:.1f}%" for a in acc_after_task]}')
        print(f'    ACC = {ACC:.2f}%')
        print(f'    BWT = {BWT:.2f}%')

        seed_accs.append(ACC)
        seed_bwts.append(BWT)

    mean_acc = np.mean(seed_accs)
    std_acc = np.std(seed_accs)
    mean_bwt = np.mean(seed_bwts)
    std_bwt = np.std(seed_bwts)

    all_results[method_name] = {
        'ACC_mean': mean_acc, 'ACC_std': std_acc,
        'BWT_mean': mean_bwt, 'BWT_std': std_bwt,
        'seed_accs': seed_accs, 'seed_bwts': seed_bwts
    }

    print(f'\n  {method_name} Final:')
    print(f'    ACC = {mean_acc:.2f} +/- {std_acc:.2f}%')
    print(f'    BWT = {mean_bwt:.2f} +/- {std_bwt:.2f}%')

# ---- Final comparison ----
print('\n' + '='*70)
print('  FINAL RESULTS (NC-TCN-20K, 21,689 params)')
print('='*70)
print(f'{"Method":<20} {"ACC":>12} {"BWT":>12} {"Params":>10}')
print('-'*56)

print(f'{"DE-KWS (paper)":<20} {"89.24%":>12} {"-3.40%":>12} {"64,480":>10}')
print(f'{"AnalyticKWS (paper)":<20} {"89.51%":>12} {"-3.20%":>12} {"~65,000":>10}')
print('-'*56)

for name, r in all_results.items():
    acc_str = f'{r["ACC_mean"]:.2f}+/-{r["ACC_std"]:.1f}%'
    bwt_str = f'{r["BWT_mean"]:.2f}+/-{r["BWT_std"]:.1f}%'
    print(f'{name:<20} {acc_str:>12} {bwt_str:>12} {"21,689":>10}')

print('\nNote: Our model is 3x smaller than DE-KWS/AnalyticKWS baselines.')


# ---- Save results to JSON ----
import json as _json
_results = {
    'experiment': 'Incremental KWS Benchmark (6-task)',
    'model': 'NC-TCN-20K',
    'params': 21689,
    'seeds': N_SEEDS,
    'hyperparameters': {
        'lambda_kd': 5.0, 'kd_temp': 3.0,
        'rehearsal_per_class': 30, 'old_aug_factor': 4,
        'lora_rank': 2, 'lora_alpha': 4, 'lr': 1.5e-3,
        'stage1_epochs': 5, 'stage2_epochs': 20
    },
    'finetune': ft_results if 'ft_results' in dir() else {},
    'ncopal': opal_results if 'opal_results' in dir() else {},
}
_save_path = os.path.join(REPO, 'results', 'kws_benchmark_latest.json')
os.makedirs(os.path.dirname(_save_path), exist_ok=True)
with open(_save_path, 'w') as _f:
    _json.dump(_results, _f, indent=2)
print(f'Results saved to {_save_path}')

# Push to GitHub
subprocess.run(['git', '-C', REPO, 'add', 'results/'], check=True)
subprocess.run(['git', '-C', REPO, 'commit', '-m', 'Auto-save KWS benchmark results'], check=False)
subprocess.run(['git', '-C', REPO, 'push'], check=False)
print('Results pushed to GitHub!')
print('Upper bound (Joint): ~95.7% ACC')

In [ ]:
# ==============================================================================
# Cell 3: Visualization
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

methods = list(all_results.keys())
colors = {'Finetune': '#EA4335', 'NC-OPAL': '#34A853'}

# Reference lines
ref_methods = ['DE-KWS\n(64K)', 'AnalyticKWS\n(65K)']
ref_accs = [89.24, 89.51]
ref_bwts = [-3.40, -3.20]

# ACC comparison
all_names = ref_methods + [f'{m}\n(21K)' for m in methods]
all_accs = ref_accs + [all_results[m]['ACC_mean'] for m in methods]
all_acc_stds = [0, 0] + [all_results[m]['ACC_std'] for m in methods]
bar_colors = ['#AAAAAA', '#AAAAAA'] + [colors.get(m, '#4285F4') for m in methods]

bars1 = ax1.bar(all_names, all_accs, yerr=all_acc_stds, color=bar_colors, capsize=5, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('ACC (%)')
ax1.set_title('Average Incremental Accuracy (higher = better)')
ax1.set_ylim(0, 100)
ax1.axhline(y=95.7, color='gold', linestyle='--', alpha=0.5, label='Joint upper bound')
for bar, val in zip(bars1, all_accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}', ha='center', fontsize=9)
ax1.legend()

# BWT comparison
all_bwts = ref_bwts + [all_results[m]['BWT_mean'] for m in methods]
all_bwt_stds = [0, 0] + [all_results[m]['BWT_std'] for m in methods]
bars2 = ax2.bar(all_names, all_bwts, yerr=all_bwt_stds, color=bar_colors, capsize=5, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('BWT (%)')
ax2.set_title('Backward Transfer (closer to 0 = less forgetting)')
ax2.axhline(y=0, color='black', linewidth=0.5)
for bar, val in zip(bars2, all_bwts):
    offset = -1.5 if val < 0 else 0.5
    ax2.text(bar.get_x() + bar.get_width()/2, val + offset, f'{val:.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: benchmark_results.png')
